# 04 · As 15 perguntas — Q1 a Q8 e QA1 a QA7

**Tech Challenge Fase 4 · POSTECH DTAT · Grupo 12**

## Objetivo
Responder, uma a uma, as **15 perguntas do escopo fechado** (`method/definitions/escopo_15_perguntas.md`):
as 8 sugeridas pelo enunciado e as 7 propostas pelo grupo. Cada pergunta traz **método declarado**
e **resultado**, e nenhuma resposta é publicada sem base e *n*.

Este notebook **não recria camada de dados**. Ele **consome** o que `src/prepara_dados.py` já escreveu.
Recriar a base aqui abriria uma segunda fonte de verdade — é exatamente o que o projeto evita.

## Entradas
| Arquivo | Papel |
|---|---|
| `00_data/gold/base_analitica.csv` | **base padrão**, n = 1.348, com os derivados prontos |
| `00_data/gold/reviews.csv` | 9.255 reviews distintos, deduplicados por `review_id` (T8) |
| `00_data/silver/produtos.csv` | **uso auxiliar e declarado**: só para dimensionar o corpus (Q7/QA5), porque o gold não carrega a lista de `review_id` por produto |
| `src/paleta.py` | paleta v06 aplicada por ferramenta (T9) |

## Saídas
| Arquivo | Conteúdo |
|---|---|
| `02_results/tables/tab13_respostas_15_perguntas.csv` | pergunta · método · resultado · base · n · situação |
| `02_results/images/fig_an_02_satisfacao_categoria_q4.png` | satisfação por categoria acima do piso (Q4) |
| `02_results/images/fig_an_03_pareto_qa2.png` | curva de Pareto das avaliações (QA2) |
| `02_results/images/fig_an_04_dispersao_subcategoria_qa3.png` | dispersão de notas por subcategoria (QA3) |
| `02_results/images/fig_an_05_ranking_wilson_qa6.png` | ranking bruto × corrigido por Wilson (QA6) |

O prefixo `fig_an_` existe para **não colidir** com `fig01`–`fig13`, geradas por `src/gera_figuras.py`.
Nenhum arquivo daquela série é sobrescrito aqui.

## Regras que governam cada célula abaixo
`T1` ranking por nota usa `nota_corrigida` (Wilson) com o *n* ao lado ·
`T2` nenhum grupo agregado com n < 30 é ranqueado — listas nominais de produtos são exceção declarada ·
`T3` todo número declara base e *n* ·
`T4` **vedada linguagem causal** — correlação é associação ·
`T5` magnitude sempre junto com significância ·
`T6` `rating_count` é **volume de avaliações**, nunca venda, receita ou demanda ·
`T7` `user_id` e `user_name` não existem a partir do silver ·
`T8` corpus declara unidade, deduplica por `review_id` e cita o truncamento em 8 ·
`T9` paleta v06 aplicada por módulo, no máximo 5 séries por visual.

**Moeda (L5):** valores monetários em **rupia indiana**, publicados **sem símbolo** e **sem conversão**.

## Duas perguntas dependiam de decisão de negócio, e as duas fecharam em 24/09/2026
| Pergunta | Decisão | Como fechou |
|---|---|---|
| **Q1** relevância | **D-2** — fórmula de relevância | **sem índice composto**: matriz 2×2 sobre os cortes já operacionais (volume ≥ p90 ou < mediana × nota bruta ≥ 4,3 ou < 4,0), que nomeia Âncora (41) e Cauda frágil (227) e reproduz Q5 = 19 e Q6 = 187 |
| **Q6** visibilidade | **D-1** — visibilidade é observável nesta base? | **não é**: volume de avaliações abaixo da mediana entra como *proxy declarado de exposição*. O funil tem 187 produtos e a lista acionável, **61**, depois da correção de Wilson (T1) |

Nenhuma das duas foi arbitrada por quem constrói: as células abaixo medem os componentes, e o
rótulo publicado vem da decisão registrada em `method/open_decisions.md`.

## 0 · Preparo

Primeira célula de código, obrigatória em todo notebook do projeto: a paleta entra **por módulo**.
Escolher cor à mão é descumprimento de T9 — por isso nenhuma célula adiante contém um hex literal
que não venha de `src/paleta.py`.

A raiz do repositório é resolvida subindo diretórios até encontrar `00_data/`. Assim o notebook roda
tanto a partir de `01_notebooks/` quanto da raiz, sem caminho absoluto embutido.

In [1]:
import sys; sys.path.insert(0, "../src")   # T9 — a paleta entra por ferramenta
from paleta import aplicar, confere_visual, rodape
aplicar()

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from paleta import CATEGORICA, CHAMADAS, ESTRUTURA, BASE, checar_series

warnings.filterwarnings("ignore", category=FutureWarning)

# --- raiz do repositório, resolvida por busca ------------------------------
RAIZ = Path.cwd()
while not (RAIZ / "00_data").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "00_data").exists():
    raise FileNotFoundError(
        "Não encontrei 00_data/ subindo a partir de " + str(Path.cwd())
        + ". Rode o notebook de dentro do repositório.")

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

GOLD = RAIZ / "00_data" / "gold"
SILVER = RAIZ / "00_data" / "silver"
IMG = RAIZ / "02_results" / "images"
TAB = RAIZ / "02_results" / "tables"
IMG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# Cores nomeadas a partir dos tokens — nunca escolhidas, sempre importadas.
AMBAR, AZUL, CLARA = CATEGORICA[0], CATEGORICA[1], CATEGORICA[2]
LARANJA, TEAL = CHAMADAS["atencao"], CHAMADAS["favoravel"]
DESTAQUE = CHAMADAS["destaque"]
TINTA, ARDOSIA, GRADE = ESTRUTURA["tinta"], ESTRUTURA["secundario"], ESTRUTURA["grade"]

print("raiz     :", RAIZ.name + "/")
print("paleta   : v06 aplicada (T9)")

raiz     : Techchallenge4-G12/
paleta   : v06 aplicada (T9)


In [2]:
def carrega(caminho, **kw):
    """Leitura com erro legível — um caminho errado precisa falhar dizendo o que falta."""
    try:
        return pd.read_csv(caminho, **kw)
    except FileNotFoundError as erro:
        raise FileNotFoundError(
            f"{caminho} não existe. Rode antes:  python3 src/prepara_dados.py") from erro


an = carrega(GOLD / "base_analitica.csv")
rev = carrega(GOLD / "reviews.csv")

# Base auxiliar, e só para o corpus: o gold não carrega a lista de review_id por
# produto (ela é grande e não serve ao dashboard). O uso está declarado no cabeçalho.
silver = carrega(SILVER / "produtos.csv")
# Texto livre vive em arquivo próprio (LGPD): juntar por product_id.
silver = silver.merge(carrega(SILVER / "produtos_texto.csv")
                      .drop(columns=["review_id"], errors="ignore"),
                      on="product_id", how="left")

N = len(an)
P90 = an.avaliacoes.quantile(0.90)
MEDIANA = an.avaliacoes.median()
PISO = 30                      # T2 — piso amostral de grupo agregado

FONTE = ("Fonte: Kaggle · karkavelrajaj/amazon-sales-dataset · base analítica n = "
         + f"{N:,}".replace(",", ".")
         + " · valores monetários em rupia indiana, sem conversão (L5)")

assert N == 1348, f"base analítica com {N} linhas — deveria ter 1.348"
assert not {"user_id", "user_name"} & set(an.columns), "T7: coluna pessoal na base"

print(f"base analítica     : {N} produtos × {an.shape[1]} colunas")
print(f"reviews distintos  : {len(rev)} (deduplicados por review_id — T8)")
print(f"corte p90          : {P90:,.0f}".replace(",", ".") + "  (engajamento alto)")
print(f"mediana            : {MEDIANA:,.0f}".replace(",", ".") + "  (engajamento baixo abaixo daqui)")

base analítica     : 1348 produtos × 17 colunas
reviews distintos  : 9255 (deduplicados por review_id — T8)
corte p90          : 36.384  (engajamento alto)
mediana            : 4.740  (engajamento baixo abaixo daqui)


### Registro das respostas

Cada pergunta respondida chama `responde(...)` uma única vez. O registro vira, no fim,
`tab13_respostas_15_perguntas.csv`. Escrever a tabela à mão permitiria divergência entre o
que o código mediu e o que a tabela publica — aqui as duas coisas são o mesmo objeto.

A coluna **situação** usa três rótulos apenas:
`respondida` · `consolidação` (Q8, que sintetiza as outras). O status
    `respondida com pendência` existe no vocabulário e **não é mais usado por nenhuma
    pergunta**: as decisões que o justificavam fecharam em 24/09/2026 ·
`consolidação` (a resposta é a leitura conjunta das demais).

In [3]:
REGISTRO = []


def responde(pergunta, metodo, resultado, base, n, situacao):
    """Guarda uma resposta no registro que será exportado como tab10."""
    REGISTRO.append({"pergunta": pergunta, "metodo": metodo, "resultado": resultado,
                     "base": base, "n": n, "situacao": situacao})
    print(f"[{pergunta}] {situacao}\n    {resultado}\n")


def salva(fig, nome):
    """Grava em 02_results/images/ com o prefixo fig_an_."""
    assert nome.startswith("fig_an_"), "prefixo obrigatório: fig_an_ (não sobrescrever fig01–fig13)"
    confere_visual(fig, nome)          # T3 e T9 conferidos na figura, não no código
    destino = IMG / nome
    fig.savefig(destino, dpi=150, bbox_inches="tight", pad_inches=0.28,
                facecolor=BASE["canvas"])
    plt.close(fig)
    print("figura:", destino.relative_to(RAIZ))


def curto(nome, tamanho=40):
    """Nome de produto encurtado — rótulo de eixo não comporta 200 caracteres."""
    nome = str(nome)
    return nome if len(nome) <= tamanho else nome[:tamanho - 1].rstrip() + "…"


def pt_br(x, casas=0):
    """Número no padrão brasileiro: ponto de milhar, vírgula decimal."""
    return f"{x:,.{casas}f}".replace(",", "§").replace(".", ",").replace("§", ".")

---
# Bloco 1 · As 8 perguntas do enunciado

---
## Q1 — Quais categorias e produtos apresentam maior relevância?

**Situação: respondida. D-2 fechada em 24/09/2026.**

"Relevância" não tinha definição fechada quando este notebook foi escrito. A decisão **D-2** — se
relevância vira índice composto ou permanece leitura multidimensional — era de Ricardo, e foi tomada:
**não se cria índice composto.** Vale a matriz 2×2 (volume × nota) para produtos e três colunas para
categorias, exatamente a recomendação que já estava registrada.

**Por que não criar o índice foi a escolha certa, e não só a mais cautelosa:** um índice ponderado
convida à pergunta *"por que esses pesos?"*, e não há resposta defensável — qualquer peso seria
arbitrado aqui e viraria número oficial por acidente. A matriz não arbitra nada: é contagem sobre
cortes que o projeto já publicava.

**Método:**
- *categorias* — apenas as de nível 1 **acima do piso** (T2): produtos, nota corrigida mediana (T1)
  e participação no total de avaliações (T6 — volume, nunca venda);
- *produtos* — matriz 2×2 de volume (cortes p90 e mediana) contra nota corrigida, que é a
  recomendação de D-2 já medida.

In [4]:
# --- componente por categoria (só acima do piso — T2) ----------------------
cat = (an.groupby("categoria_n1")
         .agg(produtos=("product_id", "size"),
              nota_corrigida_mediana=("nota_corrigida", "median"),
              avaliacoes_total=("avaliacoes", "sum"))
         .sort_values("produtos", ascending=False))
cat["participacao_avaliacoes_%"] = 100 * cat.avaliacoes_total / an.avaliacoes.sum()
cat["acima_do_piso"] = cat.produtos >= PISO

q1_cat = cat[cat.acima_do_piso].round(3)
suprimidas = cat[~cat.acima_do_piso]

print("Q1 · componentes por categoria de nível 1 — apenas grupos com n ≥ 30 (T2)")
display(q1_cat.drop(columns="acima_do_piso"))
print(f"suprimidas por T2: {len(suprimidas)} categorias somando "
      f"{int(suprimidas.produtos.sum())} produtos — nunca ranqueadas")

Q1 · componentes por categoria de nível 1 — apenas grupos com n ≥ 30 (T2)


,produtos,nota_corrigida_mediana,avaliacoes_total,participacao_avaliacoes_%
categoria_n1,,,,
Electronics,490,4.076,14208441.0,59.696
Home&Kitchen,447,3.975,2990077.0,12.563
Computers&Accessories,373,4.150,6335182.0,26.617
OfficeProducts,31,4.248,149675.0,0.629


suprimidas por T2: 5 categorias somando 7 produtos — nunca ranqueadas


In [5]:
# --- componente por produto: a matriz 2×2 adotada por D-2 (fechada 24/09/2026) ---
# Volume (≥ p90 · < mediana) × nota BRUTA (≥ 4,3 · < 4,0): os mesmos cortes que Q5 e Q6 já
# usavam. Nenhum peso é arbitrado — é contagem. A matriz reproduz Q5 = 19 e Q6 = 187 e dá nome
# aos dois quadrantes que faltavam. A nota corrigida (Wilson) ordena DENTRO de cada quadrante.
p90_vol = an.avaliacoes.quantile(0.90)
med_vol = an.avaliacoes.median()
quadrantes = {
    "Âncora (volume alto · nota ≥ 4,3)": int(((an.avaliacoes >= p90_vol) & (an.nota >= 4.3)).sum()),
    "Risco de vitrine (volume alto · nota < 4,0)": int(((an.avaliacoes >= p90_vol) & (an.nota < 4.0)).sum()),
    "Subexplorado (volume < mediana · nota ≥ 4,3)": int(((an.avaliacoes < med_vol) & (an.nota >= 4.3)).sum()),
    "Cauda frágil (volume < mediana · nota < 4,0)": int(((an.avaliacoes < med_vol) & (an.nota < 4.0)).sum()),
}
matriz = pd.Series(quadrantes, name="produtos").to_frame()
print(f"Q1 · matriz de relevância D-2 (volume: p90 = {pt_br(p90_vol, 0)} · mediana = {pt_br(med_vol, 0)})")
display(matriz)
assert list(quadrantes.values()) == [41, 19, 187, 227], f"a matriz D-2 deveria dar 41 · 19 · 187 · 227: {quadrantes}"

responde(
    "Q1",
    "componentes publicados separadamente: por categoria (produtos · nota corrigida mediana · "
    "participação no volume de avaliações, só grupos n≥30) e por produto (matriz 2×2 de D-2: volume ≥ p90 "
    "ou < mediana × nota bruta ≥ 4,3 ou < 4,0). Nenhum índice composto — D-2 fechada em 24/09/2026 nesta forma",
    f"4 categorias acima do piso concentram 1.341 produtos (99,5%); Electronics lidera em portfólio "
    f"(490) e OfficeProducts tem a maior nota corrigida mediana ({pt_br(q1_cat.nota_corrigida_mediana.max(), 3)}). "
    f"Entre produtos: Âncora 41 · Risco de vitrine 19 · Subexplorado 187 · Cauda frágil 227",
    "analítica", N,
    "respondida")


Q1 · matriz de relevância D-2 (volume: p90 = 36.384 · mediana = 4.740)


,produtos
"Âncora (volume alto · nota ≥ 4,3)",41
"Risco de vitrine (volume alto · nota < 4,0)",19
"Subexplorado (volume < mediana · nota ≥ 4,3)",187
"Cauda frágil (volume < mediana · nota < 4,0)",227


[Q1] respondida
    4 categorias acima do piso concentram 1.341 produtos (99,5%); Electronics lidera em portfólio (490) e OfficeProducts tem a maior nota corrigida mediana (4,248). Entre produtos: Âncora 41 · Risco de vitrine 19 · Subexplorado 187 · Cauda frágil 227



---
### O componente de Q1 por produto

Os quadrantes de Q1 — **Âncora**, **Risco de vitrine (Q5)**, **Subexplorado (Q6)** e **Cauda frágil** —
são publicados em `fig06_quadrantes.png` (`src/gera_figuras.py`) e na `tab17_matriz_relevancia.csv`,
**sobre a nota bruta**, que é a definição que D-2 fechou em 24/09/2026: cortes 4,0 e 4,3 cruzados com
p90 e mediana de volume.

Uma figura deste notebook publicava a mesma pergunta sobre a **nota corrigida**. Ela foi **removida
em 31/10/2026**: duas matrizes com o nome da mesma pergunta e eixos diferentes obrigam o leitor a
decidir qual vale, e a resposta já estava decidida. A nota corrigida continua onde ela decide algo —
em todo **ranking** (T1), que é o que a correção de Wilson existe para ordenar.

---
## Q2 — Quais produtos possuem maior engajamento?

**Método:** corte fechado em **p90 de `rating_count` = 36.384**. Lista nominal de produtos —
exceção declarada de T2, que dispensa o piso de 30 e exige o *n* produto a produto.

**T6 é a regra crítica aqui:** `rating_count` é volume de avaliações. Um produto no topo desta lista
é o mais **avaliado**, não o mais vendido. A base não contém venda alguma (L4).

In [6]:
alto = an[an.engajamento == "alto"]
topo = alto.nlargest(10, "avaliacoes")[
    ["product_name", "categoria_n1", "avaliacoes", "nota", "nota_corrigida"]].copy()
topo["product_name"] = topo.product_name.map(curto)
topo = topo.rename(columns={"product_name": "produto", "categoria_n1": "categoria",
                            "avaliacoes": "volume_de_avaliacoes"})

print("Q2 · os 10 maiores volumes de avaliações — nota corrigida e n exibidos ao lado (T1)")
display(topo.round(3).reset_index(drop=True))

responde(
    "Q2",
    "corte fechado no p90 de rating_count (36.384); lista nominal com volume e nota corrigida por "
    "produto — exceção de T2 para listas de produtos",
    f"{len(alto)} produtos ({100*len(alto)/N:.1f}%) estão no engajamento alto e concentram "
    f"{100*alto.avaliacoes.sum()/an.avaliacoes.sum():.1f}% do volume de avaliações; o maior isolado "
    f"tem {pt_br(alto.avaliacoes.max())} avaliações. Volume de avaliações, nunca venda (T6)",
    "analítica", N,
    "respondida")

Q2 · os 10 maiores volumes de avaliações — nota corrigida e n exibidos ao lado (T1)


,produto,categoria,volume_de_avaliacoes,nota,nota_corrigida
0,AmazonBasics Flexible Premium HDMI Cabl…,Electronics,426973.0,4.4,4.396
1,"Amazon Basics High-Speed HDMI Cable, 6…",Electronics,426973.0,4.4,4.396
2,"Amazon Basics High-Speed HDMI Cable, 6…",Electronics,426973.0,4.4,4.396
3,boAt Bassheads 100 in Ear Wired Earphon…,Electronics,363713.0,4.1,4.095
4,boAt Bassheads 100 in Ear Wired Earphon…,Electronics,363713.0,4.1,4.095
5,boAt BassHeads 100 in-Ear Wired Headpho…,Electronics,363711.0,4.1,4.095
6,"Redmi 9 Activ (Carbon Black, 4GB RAM, 6…",Electronics,313836.0,4.1,4.094
7,"Redmi 9A Sport (Coral Green, 2GB RAM, 3…",Electronics,313836.0,4.1,4.094
8,"Redmi 9A Sport (Carbon Black, 2GB RAM,…",Electronics,313832.0,4.1,4.094
9,"Redmi 9A Sport (Coral Green, 3GB RAM, 3…",Electronics,313832.0,4.1,4.094


[Q2] respondida
    136 produtos (10.1%) estão no engajamento alto e concentram 61.6% do volume de avaliações; o maior isolado tem 426.973 avaliações. Volume de avaliações, nunca venda (T6)



---
## Q3 — Existe relação entre descontos e avaliações?

**Método:** correlação de **Spearman** (monotônica, sem exigir linearidade nem normalidade) entre
`desconto` e `nota`, sobre a base analítica inteira.

**T5 obriga publicar magnitude junto com significância.** Com n = 1.348, um ρ pequeno cruza o limiar
de significância com folga — dizer só "é significativo" seria enganoso.
**T4 veda a leitura causal:** o que se mede é associação.

In [7]:
from scipy.stats import spearmanr

PARES = [("Desconto × nota", "desconto", "nota"),
         ("Desconto × engajamento", "desconto", "avaliacoes"),
         ("Preço de tabela × nota", "preco_tab", "nota"),
         ("Engajamento × nota", "avaliacoes", "nota")]

linhas = []
for rotulo, a, b in PARES:
    rho, p = spearmanr(an[a], an[b])
    leitura = ("sem associação" if p >= 0.05 else
               ("significativa, magnitude pequena" if abs(rho) < 0.3 else
                "significativa, magnitude moderada"))
    linhas.append({"par": rotulo, "rho": round(rho, 4), "p_valor": float(f"{p:.3g}"),
                   "n": N, "leitura": leitura})

correl = pd.DataFrame(linhas)
print("As 4 associações de Spearman — magnitude e significância lado a lado (T5)")
display(correl)
print("Nenhuma destas linhas afirma causa. Associação monotônica é o que a medida entrega (T4).")

r_desc_nota = correl.loc[0, "rho"]; p_desc_nota = correl.loc[0, "p_valor"]
r_desc_eng = correl.loc[1, "rho"]; p_desc_eng = correl.loc[1, "p_valor"]
r_preco_nota = correl.loc[2, "rho"]; p_preco_nota = correl.loc[2, "p_valor"]
r_eng_nota = correl.loc[3, "rho"]; p_eng_nota = correl.loc[3, "p_valor"]

responde(
    "Q3",
    "correlação de Spearman entre desconto e nota, e entre desconto e volume de avaliações; "
    "magnitude publicada junto ao p-valor (T5)",
    f"associação negativa e pequena entre desconto e nota (ρ = {pt_br(r_desc_nota, 3)}, "
    f"p = {p_desc_nota:.1e}) e entre desconto e volume de avaliações (ρ = {pt_br(r_desc_eng, 3)}, "
    f"p = {p_desc_eng:.1e}). Significativas e pequenas ao mesmo tempo — associação, não causa (T4)",
    "analítica", N,
    "respondida")

As 4 associações de Spearman — magnitude e significância lado a lado (T5)


,par,rho,p_valor,n,leitura
0,Desconto × nota,-0.1500,3.140000e-08,1348,"significativa, magnitude pequena"
1,Desconto × engajamento,-0.1322,1.120000e-06,1348,"significativa, magnitude pequena"
2,Preço de tabela × nota,0.0323,2.360000e-01,1348,sem associação
3,Engajamento × nota,0.1933,8.290000e-13,1348,"significativa, magnitude pequena"


Nenhuma destas linhas afirma causa. Associação monotônica é o que a medida entrega (T4).
[Q3] respondida
    associação negativa e pequena entre desconto e nota (ρ = -0,150, p = 3.1e-08) e entre desconto e volume de avaliações (ρ = -0,132, p = 1.1e-06). Significativas e pequenas ao mesmo tempo — associação, não causa (T4)



---
## Q4 — Quais categorias apresentam maior ou menor satisfação?

**Método:** nota corrigida de Wilson (T1) por categoria de nível 1, **só nos 4 grupos com n ≥ 30** (T2).
As outras 5 categorias somam 7 produtos e ficam fora de qualquer ranqueamento — suprimidas, não omitidas:
a supressão é declarada no rodapé da figura (L3).

Publicar a mediana da nota corrigida, e não a média da nota bruta, evita que um grupo pequeno com
notas extremas apareça no topo por variância.

In [8]:
q4 = (an[an.categoria_n1.isin(q1_cat.index)]
        .groupby("categoria_n1")
        .agg(produtos=("product_id", "size"),
             nota_mediana=("nota", "median"),
             nota_corrigida_mediana=("nota_corrigida", "median"),
             pct_satisfacao_baixa=("satisfacao", lambda s: 100 * (s == "baixa").mean()))
        .sort_values("nota_corrigida_mediana", ascending=False)
        .round(3))
print("Q4 · satisfação por categoria — apenas os 4 grupos acima do piso de 30 (T2)")
display(q4)

maior, menor = q4.index[0], q4.index[-1]
responde(
    "Q4",
    "mediana da nota corrigida de Wilson por categoria de nível 1 (T1), restrita aos grupos com "
    "n ≥ 30 (T2); 5 categorias com 7 produtos ao todo ficam suprimidas (L3)",
    f"{maior} tem a maior nota corrigida mediana ({pt_br(q4.nota_corrigida_mediana.iloc[0], 3)}, "
    f"n = {int(q4.produtos.iloc[0])}) e {menor} a menor ({pt_br(q4.nota_corrigida_mediana.iloc[-1], 3)}, "
    f"n = {int(q4.produtos.iloc[-1])}); a amplitude entre as quatro é de "
    f"{pt_br(q4.nota_corrigida_mediana.max() - q4.nota_corrigida_mediana.min(), 3)} ponto de nota",
    "analítica", int(q4.produtos.sum()),
    "respondida")

Q4 · satisfação por categoria — apenas os 4 grupos acima do piso de 30 (T2)


,produtos,nota_mediana,nota_corrigida_mediana,pct_satisfacao_baixa
categoria_n1,,,,
OfficeProducts,31,4.3,4.248,0.000
Computers&Accessories,373,4.2,4.150,17.426
Electronics,490,4.1,4.076,26.122
Home&Kitchen,447,4.1,3.975,32.215


[Q4] respondida
    OfficeProducts tem a maior nota corrigida mediana (4,248, n = 31) e Home&Kitchen a menor (3,975, n = 447); a amplitude entre as quatro é de 0,273 ponto de nota



In [9]:
checar_series(2)

fig, ax = plt.subplots(figsize=(8.6, 4.0))
ordem = q4.sort_values("nota_corrigida_mediana")
y = np.arange(len(ordem))

ax.barh(y, ordem.nota_corrigida_mediana, color=TEAL, height=0.52,
        label="nota corrigida mediana (Wilson · T1)")
ax.scatter(ordem.nota_mediana, y, color=DESTAQUE, s=64, zorder=5,
           edgecolors=BASE["card"], linewidth=1.0, label="nota bruta mediana")

for i, (nome, linha) in enumerate(ordem.iterrows()):
    ax.text(linha.nota_corrigida_mediana - 0.02, i,
            pt_br(linha.nota_corrigida_mediana, 3), va="center", ha="right",
            fontsize=8.5, color=BASE["card"], fontweight="bold")
    ax.text(0.03, i + 0.30, f"n = {int(linha.produtos)} produtos · "
            f"{pt_br(linha.pct_satisfacao_baixa, 1)}% com nota < 4,0",
            va="center", fontsize=7.6, color=ARDOSIA)

ax.set_yticks(y); ax.set_yticklabels([curto(i, 26) for i in ordem.index])
ax.set_xlim(0, 4.6)
# Folga no topo para a anotação de n da última categoria não encostar no título.
ax.set_ylim(-0.6, len(ordem) - 0.28)
ax.set_xlabel("nota (escala 1 a 5)")
ax.set_title("Q4 · Satisfação por categoria de nível 1 — 4 grupos acima do piso amostral", pad=10)
# Legenda fora da área: dentro dela cobriria a barra de Home&Kitchen e seu rótulo.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=8)
rodape(fig, FONTE + "\nT2: 5 categorias (Musical Instruments, Home Improvement, Toys & Games, "
       "Health & Personal Care, Car & Motorbike) somam 7 produtos e foram suprimidas — nunca ranqueadas (L3)")
salva(fig, "fig_an_02_satisfacao_categoria_q4.png")

figura: 02_results/images/fig_an_02_satisfacao_categoria_q4.png


---
## Q5 — Existem produtos com alto engajamento e baixa avaliação?

**Método:** cruzamento dos dois cortes fechados — `rating_count ≥ 36.384` (p90) **e** `rating < 4,0`.
Lista nominal: T2 não se aplica, e o *n* de avaliações aparece produto a produto.

São **19 produtos**: pouco em contagem, muito em exposição. É o quadrante que o projeto chama de
**risco de vitrine** — produtos muito avaliados — volume no decil superior, proxy declarado de exposição (D-1) — e com nota baixa.
Nada aqui afirma que a exposição *causa* a nota baixa (T4).

In [10]:
q5 = an[an.q5_risco_vitrine].sort_values("avaliacoes", ascending=False)
lista_q5 = q5[["product_name", "categoria_n1", "avaliacoes", "nota", "nota_corrigida", "desconto"]].copy()
lista_q5["product_name"] = lista_q5.product_name.map(curto)
lista_q5 = lista_q5.rename(columns={"product_name": "produto", "categoria_n1": "categoria",
                                    "avaliacoes": "volume_de_avaliacoes"})
print(f"Q5 · os {len(q5)} produtos do quadrante de risco de vitrine — lista nominal (exceção de T2)")
display(lista_q5.round(3).reset_index(drop=True))

responde(
    "Q5",
    "cruzamento dos cortes fechados: rating_count ≥ p90 (36.384) e rating < 4,0; lista nominal com "
    "o n de avaliações por produto (exceção de T2)",
    f"sim — {len(q5)} produtos, {100*len(q5)/N:.1f}% da base, concentrando "
    f"{pt_br(q5.avaliacoes.sum())} avaliações ({100*q5.avaliacoes.sum()/an.avaliacoes.sum():.1f}% do total); "
    f"nota mediana {pt_br(q5.nota.median(), 2)} e desconto mediano {pt_br(q5.desconto.median(), 0)}",
    "analítica", int(len(q5)),
    "respondida")

Q5 · os 19 produtos do quadrante de risco de vitrine — lista nominal (exceção de T2)


,produto,categoria,volume_de_avaliacoes,nota,nota_corrigida,desconto
0,boAt Airdopes 121v2 in-Ear True Wireles…,Electronics,180998.0,3.8,3.792,57
1,boAt Airdopes 141 Bluetooth Truly Wirel…,Electronics,136954.0,3.9,3.891,67
2,Pigeon by Stovekraft Amaze Plus Electri…,Home&Kitchen,123365.0,3.9,3.890,48
3,Boult Audio Probass Curve Bluetooth Wir…,Electronics,103052.0,3.8,3.789,80
4,"JBL C50HI, Wired in Ear Headphones with…",Electronics,92995.0,3.9,3.888,50
5,"JBL Tune 215BT, 16 Hrs Playtime with Qu…",Electronics,87798.0,3.7,3.688,50
6,PTron Tangent Lite Bluetooth 5.0 Earpho…,Electronics,83996.0,3.5,3.487,67
7,Boult Audio BassBuds X1 in-Ear Wired Ea…,Electronics,77027.0,3.9,3.887,67
8,Zebronics ZEB-COUNTY 3W Wireless Blueto…,Electronics,64705.0,3.9,3.886,45
9,Prestige Electric Kettle PKOSS - 1500wa…,Home&Kitchen,63350.0,3.9,3.886,48


[Q5] respondida
    sim — 19 produtos, 1.4% da base, concentrando 1.453.389 avaliações (6.1% do total); nota mediana 3,90 e desconto mediano 57



### Apoio de ML2 — quais atributos acompanham a baixa satisfação

O registro de ML (`method/definitions/registro_machine_learning.md`) define ML2 como **ranking de
importância de atributos**, não como probabilidade de risco por produto. A especificação é obrigatória
e está reproduzida abaixo na íntegra, porque sem ela o número não é verificável:
Random Forest de 300 árvores, `random_state=42`, numéricos padronizados, categoria nível 1 em *one-hot*,
`StratifiedGroupKFold(5, shuffle=True, random_state=42)`, **agrupada pelo par `(avaliacoes, nota)`** — a regra de
variante da L8.

**O agrupamento não é detalhe.** A L8 declara que 254 produtos são listagens-gêmeas, com exatamente o mesmo par
`(rating_count, rating)`. O alvo é `nota < 4,0` e `avaliacoes` é preditor: com a validação ingênua, uma gêmea no
treino entrega ao modelo o rótulo da gêmea no teste. Medido: o AUC do modelo completo cai de 0,750 para 0,704.
A validação mora em `src/ml2.py`, a mesma do relatório.

**O embaralhamento também não é detalhe.** O CSV vem ordenado; sem `shuffle=True` o `StratifiedKFold` cai para ≈0,33.

**Vedação de vazamento:** nenhum preditor deriva de `rating`. **Leitura permitida:** importância de
atributo indica associação, nunca causa (T4).

In [11]:
import sys
sys.path.insert(0, "../src")
import ml2
from sklearn.preprocessing import StandardScaler   # usado adiante, no QA7

NUM = ["preco_tab", "preco_desc", "desconto", "avaliacoes"]   # nenhum deriva de rating
CAT = ["categoria_n1"]
alvo = ml2.alvo(an)                          # nota < 4,0 — mesmo limiar dos cortes operacionais

blocos = [("volume", ["avaliacoes"], []),
          ("comerciais", ["preco_tab", "preco_desc", "desconto"], []),
          ("comerciais + categoria", ["preco_tab", "preco_desc", "desconto"], CAT),
          ("completo", NUM, CAT)]

ml2_tab = []
for nome, num, categ in blocos:
    agrupada = ml2.auc(an, num, categ, agrupar=True)      # a que se publica
    ingenua = ml2.auc(an, num, categ, agrupar=False)      # só para medir o vazamento
    ml2_tab.append({"bloco_preditores": nome, "auc": round(agrupada.mean(), 3),
                    "desvio_entre_particoes": round(agrupada.std(), 3),
                    "auc_validacao_ingenua": round(ingenua.mean(), 3)})
ml2_tab = pd.DataFrame(ml2_tab)
print("ML2 · AUC por bloco de preditores — validação cruzada agrupada por variante (L8), taxa de risco 25,1%")
display(ml2_tab)

modelo = ml2.monta(NUM, CAT).fit(an[NUM + CAT], alvo)
nomes = list(NUM) + list(modelo.named_steps["pre"].named_transformers_["c"].get_feature_names_out(CAT))
imp = (pd.Series(modelo.named_steps["rf"].feature_importances_, index=nomes)
         .sort_values(ascending=False).head(6).round(4))
print("\nML2 · ranking de importância de atributos (associação, nunca causa — T4)")
display(imp.rename("importancia").to_frame())
print("Não existe coluna de probabilidade de risco por produto: a saída de ML2 é este ranking.")


ML2 · AUC por bloco de preditores — validação cruzada agrupada por variante (L8), taxa de risco 25,1%


,bloco_preditores,auc,desvio_entre_particoes,auc_validacao_ingenua
0,volume,0.556,0.038,0.622
1,comerciais,0.583,0.029,0.624
2,comerciais + categoria,0.658,0.026,0.691
3,completo,0.704,0.006,0.750



ML2 · ranking de importância de atributos (associação, nunca causa — T4)


,importancia
avaliacoes,0.3015
desconto,0.2215
preco_desc,0.2203
preco_tab,0.2082
categoria_n1_Computers&Accessories,0.0170
categoria_n1_Home&Kitchen,0.0132


Não existe coluna de probabilidade de risco por produto: a saída de ML2 é este ranking.


---
## Q6 — Existem produtos com boa avaliação e pouca visibilidade?

**Situação: respondida. D-1 fechada em 24/09/2026.**

A pergunta usa "visibilidade", e **T6 separa os dois termos**: engajamento é o rótulo de negócio de
`rating_count`; visibilidade é conceito distinto. A base **não tem** impressão, clique ou posição de
busca — visibilidade real não é observável aqui. **D-1 fechou adotando volume de avaliações abaixo da
mediana como *proxy declarado de exposição*** — declarado justamente porque não é visibilidade medida.
Fica **vedado** escrever "pouca visibilidade" em qualquer visual: o rótulo publicado é *"bem avaliados
com baixo volume de avaliações"*.

**Duas contagens, e a segunda é a que vale para recomendar.** O cruzamento `rating ≥ 4,3` **e**
`rating_count < 4.740` devolve o **funil**. Mas T1 exige nota corrigida em todo ranking — e uma lista
de oportunidades é um ranking. Aplicando Wilson, a maior parte do funil cai: o que sobra é a **lista
acionável**, e é ela que vai para o relatório e para o dashboard (medida M22).

In [12]:
q6 = an[an.q6_subexplorado].sort_values("nota_corrigida", ascending=False)
print(f"Q6 · conjunto candidato: {len(q6)} produtos com nota ≥ 4,3 e volume abaixo da mediana")
print(f"     mediana de avaliações no grupo: {pt_br(q6.avaliacoes.median())} "
      f"(base inteira: {pt_br(MEDIANA)})")
print(f"     nota corrigida mediana: {pt_br(q6.nota_corrigida.median(), 3)}")

amostra = q6.head(10)[["product_name", "categoria_n1", "avaliacoes", "nota", "nota_corrigida"]].copy()
amostra["product_name"] = amostra.product_name.map(curto)
amostra = amostra.rename(columns={"product_name": "produto", "categoria_n1": "categoria",
                                  "avaliacoes": "volume_de_avaliacoes"})
print("\n  os 10 de maior nota corrigida dentro do candidato (T1 — n exibido ao lado):")
display(amostra.round(3).reset_index(drop=True))

q6_acionavel = int((an.q6_subexplorado & (an.nota_corrigida >= 4.3)).sum())
caem = len(q6) - q6_acionavel
print(f"\n  T1 aplicada ao funil: dos {len(q6)}, {caem} caem abaixo de 4,3 na nota corrigida")
print(f"  -> lista acionável: {q6_acionavel} produtos (medida M22 do catálogo DAX)")
print(f"  menor volume dentro do funil: {pt_br(q6.avaliacoes.min())} avaliações "
      f"— o caso que T1 existe para barrar")

responde(
    "Q6",
    "cruzamento rating ≥ 4,3 e rating_count < mediana (4.740) como funil; D-1 fechada em "
    "24/09/2026 adotando volume abaixo da mediana como proxy declarado de exposição. "
    "T1 aplicada sobre o funil produz a lista acionável",
    f"{len(q6)} produtos ({100*len(q6)/N:.1f}%) atendem ao critério candidato, com nota corrigida "
    f"mediana {pt_br(q6.nota_corrigida.median(), 3)} e mediana de {pt_br(q6.avaliacoes.median())} "
    f"avaliações. Desses, {q6_acionavel} sobrevivem à correção de Wilson e formam a lista "
    f"acionável ({caem} caem). Vedado chamá-los de 'pouco visíveis': a base não observa "
    f"impressão nem clique (T6)",
    "analítica", N,
    "respondida")

Q6 · conjunto candidato: 187 produtos com nota ≥ 4,3 e volume abaixo da mediana
     mediana de avaliações no grupo: 1.552 (base inteira: 4.740)
     nota corrigida mediana: 4,242

  os 10 de maior nota corrigida dentro do candidato (T1 — n exibido ao lado):


,produto,categoria,volume_de_avaliacoes,nota,nota_corrigida
0,"Instant Pot Air Fryer, Vortex 2QT, Touc…",Home&Kitchen,3964.0,4.8,4.771
1,Campfire Spring Chef Prolix Instant Por…,Home&Kitchen,2591.0,4.7,4.657
2,Multifunctional 2 in 1 Electric Egg Boi…,Home&Kitchen,2300.0,4.7,4.654
3,FIGMENT Handheld Milk Frother Rechargea…,Home&Kitchen,1729.0,4.7,4.646
4,"10k 8k 4k HDMI Cable, Certified 48Gbps…",Electronics,3664.0,4.6,4.559
5,Xiaomi Pad 5| Qualcomm Snapdragon 860|…,Computers&Accessories,2886.0,4.6,4.554
6,Ikea 903.391.72 Polypropylene Plastic S…,Home&Kitchen,2781.0,4.6,4.553
7,"Sujata Dynamix, Mixer Grinder, 900 Watt…",Home&Kitchen,2751.0,4.6,4.553
8,Classmate Soft Cover 6 Subject Spiral B…,OfficeProducts,4428.0,4.5,4.460
9,Tukzer Gel Mouse Pad Wrist Rest Memory-…,Computers&Accessories,4219.0,4.5,4.459



  T1 aplicada ao funil: dos 187, 126 caem abaixo de 4,3 na nota corrigida
  -> lista acionável: 61 produtos (medida M22 do catálogo DAX)
  menor volume dentro do funil: 5 avaliações — o caso que T1 existe para barrar
[Q6] respondida
    187 produtos (13.9%) atendem ao critério candidato, com nota corrigida mediana 4,242 e mediana de 1.552 avaliações. Desses, 61 sobrevivem à correção de Wilson e formam a lista acionável (126 caem). Vedado chamá-los de 'pouco visíveis': a base não observa impressão nem clique (T6)



---
## Q7 — Quais padrões podem ser identificados nas avaliações?

**Método e fronteira.** O tratamento textual completo (n-gramas e TF-IDF, ML3) é entrega do
`05_nlp.ipynb`. Aqui a pergunta é respondida no que lhe cabe: **dimensionar e qualificar o corpus**,
que é o que torna qualquer achado textual publicável.

**T8 exige três declarações, e todas aparecem abaixo:**
1. **unidade** — o bloco de review (até 8 avaliações de um produto), rotulado pela nota do produto;
2. **deduplicação por `review_id`** — há 1.333 referências repetidas; sem isso o corpus aparece 14% maior;
3. **truncamento em 8** — média medida de 7,85 avaliações por produto.

**L1 é a limitação mais importante do projeto:** o texto não é amostra aleatória do que os consumidores
escreveram, é o recorte que o marketplace exibe. Achado de texto descreve **os temas visíveis na vitrine**.

In [13]:
# O gold não carrega a lista de review_id por produto — daí o uso auxiliar do silver,
# declarado no cabeçalho. A base analítica continua sendo o filtro (nota e volume válidos).
base_texto = silver[silver.nota.notna() & silver.avaliacoes.notna()].copy()
ref = base_texto.assign(_rid=base_texto.review_id.str.split(",")).explode("_rid")
ref["_rid"] = ref._rid.str.strip()

por_produto = ref.groupby("product_id")._rid.size()
referencias, distintos = len(ref), ref._rid.nunique()

print("Q7 · dimensionamento do corpus (T8)")
print(f"  blocos de review        : {len(base_texto)}   (1 por produto da base analítica)")
print(f"  referências de avaliação: {pt_br(referencias)}")
print(f"  reviews distintos       : {pt_br(distintos)}   ({pt_br(referencias - distintos)} repetidas)")
print(f"  truncamento             : média {pt_br(por_produto.mean(), 2)} · máximo {por_produto.max()} por produto")
print(f"  conferência com o gold  : reviews.csv tem {pt_br(len(rev))} linhas — igual aos distintos")

ref_sat = ref.merge(an[["product_id", "satisfacao"]], on="product_id")
corpus = []
for faixa, rotulo in [("baixa", "< 4,0"), ("media", "4,0–4,29"), ("alta", "≥ 4,3")]:
    sub = ref_sat[ref_sat.satisfacao == faixa]
    corpus.append({"faixa_nota": rotulo,
                   "blocos": int((an.satisfacao == faixa).sum()),
                   "referencias": len(sub),
                   "reviews_distintos": sub._rid.nunique()})
corpus = pd.DataFrame(corpus)
print("\n  as três contagens lado a lado, por faixa de nota:")
display(corpus)

responde(
    "Q7",
    "dimensionamento e qualificação do corpus segundo T8 — unidade = bloco de review, deduplicação "
    "por review_id, truncamento declarado; extração de n-gramas e TF-IDF é entrega do 05_nlp (ML3)",
    f"o corpus contrasta {corpus.blocos[0]} blocos de nota < 4,0 e {corpus.blocos[2]} de nota ≥ 4,3, "
    f"sobre {pt_br(distintos)} reviews distintos vindos de {pt_br(referencias)} referências "
    f"({pt_br(referencias - distintos)} repetidas). Truncamento em 8 (média {pt_br(por_produto.mean(), 2)}): "
    "os temas são os visíveis na vitrine, não a distribuição real das opiniões (L1)",
    "analítica · corpus deduplicado", int(distintos),
    "respondida")

Q7 · dimensionamento do corpus (T8)
  blocos de review        : 1348   (1 por produto da base analítica)
  referências de avaliação: 10.588
  reviews distintos       : 9.255   (1.333 repetidas)
  truncamento             : média 7,85 · máximo 8 por produto
  conferência com o gold  : reviews.csv tem 9.255 linhas — igual aos distintos

  as três contagens lado a lado, por faixa de nota:


,faixa_nota,blocos,referencias,reviews_distintos
0,"< 4,0",339,2652,2502
1,"4,0–4,29",591,4674,3875
2,"≥ 4,3",418,3262,2878


[Q7] respondida
    o corpus contrasta 339 blocos de nota < 4,0 e 418 de nota ≥ 4,3, sobre 9.255 reviews distintos vindos de 10.588 referências (1.333 repetidas). Truncamento em 8 (média 7,85): os temas são os visíveis na vitrine, não a distribuição real das opiniões (L1)



---
## Q8 — Quais oportunidades ou riscos para o negócio?

**Método:** consolidação. Q8 não tem medida própria — ela lê as demais e as ordena por tamanho de
exposição. Inventar número novo aqui seria criar uma décima sexta pergunta.

**Duas vedações governam a redação:**
- **L4** — a base não tem venda, receita, margem, estoque nem data. Nenhuma recomendação pode ser
  expressa em receita, volume vendido ou retorno financeiro;
- **T4** — nada aqui afirma que um fator causa o outro.

In [14]:
consolidado = pd.DataFrame([
    {"leitura": "risco", "achado": "risco de vitrine (Q5)",
     "tamanho": f"{len(q5)} produtos · {pt_br(q5.avaliacoes.sum())} avaliações",
     "exposicao_%": round(100 * q5.avaliacoes.sum() / an.avaliacoes.sum(), 1)},
    {"leitura": "risco", "achado": "desconto profundo acompanhado de nota menor (Q3/QA1)",
     "tamanho": f"faixa 81–100%: {int((an.faixa_desconto == '81-100%').sum())} produtos",
     "exposicao_%": round(100 * an[an.faixa_desconto == "81-100%"].avaliacoes.sum() / an.avaliacoes.sum(), 1)},
    {"leitura": "risco", "achado": "concentração de volume em poucos produtos (QA2)",
     "tamanho": f"{int(an.pareto_80.sum())} produtos concentram 80% das avaliações",
     "exposicao_%": 80.0},
    {"leitura": "oportunidade", "achado": "candidato a subexploração (Q6 — D-1 fechada em 24/09/2026)",
     "tamanho": f"{len(q6)} produtos com nota ≥ 4,3 e volume abaixo da mediana",
     "exposicao_%": round(100 * q6.avaliacoes.sum() / an.avaliacoes.sum(), 1)},
    {"leitura": "oportunidade", "achado": "revisão de sortimento no arquétipo B (QA7)",
     "tamanho": "335 produtos — único grupo com nota mediana abaixo de 4,0",
     "exposicao_%": np.nan},
])
print("Q8 · consolidação — ordenada por exposição em volume de avaliações, não por receita (L4)")
display(consolidado)

responde(
    "Q8",
    "consolidação das demais perguntas, ordenada por exposição em volume de avaliações; sem medida "
    "própria e sem tradução financeira (L4)",
    f"o maior risco por exposição é o quadrante de vitrine ({len(q5)} produtos com "
    f"{100*q5.avaliacoes.sum()/an.avaliacoes.sum():.1f}% do volume de avaliações) somado à concentração "
    f"de 80% do volume em {int(an.pareto_80.sum())} produtos; a maior oportunidade medida é o candidato "
    f"de {len(q6)} produtos bem avaliados e com baixo volume de avaliações (D-1)",
    "analítica", N,
    "consolidação")

Q8 · consolidação — ordenada por exposição em volume de avaliações, não por receita (L4)


,leitura,achado,tamanho,exposicao_%
0,risco,risco de vitrine (Q5),19 produtos · 1.453.389 avaliações,6.1
1,risco,desconto profundo acompanhado de nota menor (Q...,faixa 81–100%: 51 produtos,1.9
2,risco,concentração de volume em poucos produtos (QA2),306 produtos concentram 80% das avaliações,80.0
3,oportunidade,candidato a subexploração (Q6 — D-1 fechada em...,"187 produtos com nota ≥ 4,3 e volume abaixo da...",1.3
4,oportunidade,revisão de sortimento no arquétipo B (QA7),335 produtos — único grupo com nota mediana ab...,NaN


[Q8] consolidação
    o maior risco por exposição é o quadrante de vitrine (19 produtos com 6.1% do volume de avaliações) somado à concentração de 80% do volume em 306 produtos; a maior oportunidade medida é o candidato de 187 produtos bem avaliados e com baixo volume de avaliações (D-1)



---
# Bloco 2 · As 7 perguntas próprias do grupo

---
## QA1 — Existe faixa de desconto em que nota e engajamento se sustentam?

**Método:** cinco faixas fixas de desconto, cada uma com **nota corrigida** (T1) e **volume mediano de
avaliações** (T6). Todas as cinco ficam acima do piso de 30 — a menor tem 51 produtos —, então todas
podem ser comparadas (T2).

Esta pergunta absorve o antigo teste isolado de corte em 60%: em vez de perguntar se existe um limiar
mágico, mede-se a faixa inteira. **T4:** a leitura é de acompanhamento entre faixas, não de efeito.

In [15]:
ORDEM_FX = ["0-20%", "21-40%", "41-60%", "61-80%", "81-100%"]
an["faixa_desconto"] = pd.Categorical(an.faixa_desconto, ORDEM_FX, ordered=True)

qa1 = (an.groupby("faixa_desconto", observed=True)
         .agg(n=("product_id", "size"),
              nota_mediana=("nota", "median"),
              nota_corrigida_wilson=("nota_corrigida", "median"),
              avaliacoes_medianas=("avaliacoes", "median"),
              pct_satisfacao_baixa=("satisfacao", lambda s: 100 * (s == "baixa").mean()))
         .round(3))
print("QA1 · faixas de desconto — todas acima do piso de 30 (T2), a menor com 51 produtos")
display(qa1)
print("mínimo de n entre as faixas:", int(qa1.n.min()), "— nenhuma faixa suprimida")

melhor_nota = qa1.nota_corrigida_wilson.idxmax()
maior_volume = qa1.avaliacoes_medianas.idxmax()
responde(
    "QA1",
    "cinco faixas fixas de desconto comparadas por nota corrigida de Wilson (T1) e volume mediano de "
    "avaliações (T6); todas com n ≥ 30 (T2)",
    f"não há faixa em que os dois subam juntos: a nota corrigida mediana é máxima em {melhor_nota} "
    f"({pt_br(qa1.nota_corrigida_wilson.max(), 3)}) e cai até {pt_br(qa1.nota_corrigida_wilson.min(), 3)} "
    f"em 81-100%, enquanto o volume mediano é máximo em {maior_volume} "
    f"({pt_br(qa1.avaliacoes_medianas.max())}). A faixa 21-40% é a que mais se aproxima de sustentar "
    "as duas leituras — associação entre faixas, não efeito do desconto (T4)",
    "analítica", N,
    "respondida")

QA1 · faixas de desconto — todas acima do piso de 30 (T2), a menor com 51 produtos


,n,nota_mediana,nota_corrigida_wilson,avaliacoes_medianas,pct_satisfacao_baixa
faixa_desconto,,,,,
0-20%,175,4.2,4.136,6736.0,16.571
21-40%,329,4.1,4.083,7241.0,19.757
41-60%,469,4.1,4.025,3025.0,28.358
61-80%,324,4.1,4.035,4147.0,27.778
81-100%,51,4.0,3.906,4426.0,43.137


mínimo de n entre as faixas: 51 — nenhuma faixa suprimida
[QA1] respondida
    não há faixa em que os dois subam juntos: a nota corrigida mediana é máxima em 0-20% (4,136) e cai até 3,906 em 81-100%, enquanto o volume mediano é máximo em 21-40% (7.241). A faixa 21-40% é a que mais se aproxima de sustentar as duas leituras — associação entre faixas, não efeito do desconto (T4)



---
## QA2 — Quantos produtos concentram 80% das avaliações?

**Método:** curva de Pareto sobre `rating_count` ordenado em ordem decrescente; conta-se quantos
produtos são necessários para alcançar 80% do volume total. A coluna `pareto_80` do gold já carrega
essa marcação — aqui ela é **recalculada e conferida**, porque uma marcação herdada sem conferência
é um número sem prova.

In [16]:
ordenado = an.sort_values("avaliacoes", ascending=False).reset_index(drop=True)
acumulado = ordenado.avaliacoes.cumsum() / ordenado.avaliacoes.sum()
corte_pareto = int((acumulado < 0.80).sum()) + 1      # +1: o produto que cruza a linha entra

print(f"QA2 · {corte_pareto} produtos concentram 80% do volume de avaliações "
      f"({100*corte_pareto/N:.1f}% da base analítica)")
print(f"      marcação pareto_80 no gold: {int(an.pareto_80.sum())} — confere: "
      f"{corte_pareto == int(an.pareto_80.sum())}")
print(f"      os 10 maiores sozinhos respondem por {100*acumulado.iloc[9]:.1f}% do volume")

responde(
    "QA2",
    "curva de Pareto sobre rating_count em ordem decrescente, recalculada e conferida contra a "
    "marcação pareto_80 do gold",
    f"{corte_pareto} produtos — {100*corte_pareto/N:.1f}% da base analítica — concentram 80% do volume "
    f"de avaliações; os 10 maiores respondem por {100*acumulado.iloc[9]:.1f}%. Concentração de "
    "avaliações, não de vendas (T6)",
    "analítica", N,
    "respondida")

QA2 · 306 produtos concentram 80% do volume de avaliações (22.7% da base analítica)
      marcação pareto_80 no gold: 306 — confere: True
      os 10 maiores sozinhos respondem por 15.2% do volume
[QA2] respondida
    306 produtos — 22.7% da base analítica — concentram 80% do volume de avaliações; os 10 maiores respondem por 15.2%. Concentração de avaliações, não de vendas (T6)



In [17]:
checar_series(2)

fig, ax = plt.subplots(figsize=(8.6, 4.3))
x = np.arange(1, N + 1)
ax.plot(x, 100 * acumulado, color=TEAL, lw=2.1, label="acumulado do volume de avaliações")
ax.fill_between(x[:corte_pareto], 100 * acumulado[:corte_pareto], color=TEAL, alpha=0.14)

ax.axhline(80, color=GRADE, ls="--", lw=1.1)
ax.axvline(corte_pareto, color=LARANJA, ls="--", lw=1.4)
ax.annotate(f"{corte_pareto} produtos = {100*corte_pareto/N:.1f}% da base\nconcentram 80% das avaliações",
            xy=(corte_pareto, 80), xytext=(corte_pareto + 150, 52), fontsize=9, color=TINTA,
            arrowprops=dict(arrowstyle="->", color=LARANJA, lw=1.3))

ax.set_xlabel("produtos ordenados por volume de avaliações (decrescente)")
ax.set_ylabel("% acumulado do volume de avaliações")
ax.set_ylim(0, 102); ax.set_xlim(0, N)
ax.set_title("QA2 · Curva de Pareto — concentração do volume de avaliações")
ax.legend(loc="lower right", fontsize=8)
rodape(fig, FONTE + "\nrating_count é volume de avaliações — nunca vendas, receita ou demanda (T6 · L4)")
salva(fig, "fig_an_03_pareto_qa2.png")

figura: 02_results/images/fig_an_03_pareto_qa2.png


---
## QA3 — Quais subcategorias (nível 2) têm maior dispersão de notas?

**Método:** desvio-padrão e IQR da nota, por subcategoria de nível 2, **só nos grupos com n ≥ 30** (T2).
São 8 grupos, cobrindo 88,3% dos produtos — a cobertura é declarada porque o que sobra não é pouco.

Dispersão alta significa **portfólio heterogêneo** dentro da mesma prateleira: a mediana da
subcategoria esconde produtos muito diferentes entre si.

In [18]:
cont2 = an.categoria_n2.value_counts()
acima2 = cont2[cont2 >= PISO].index

qa3 = (an[an.categoria_n2.isin(acima2)]
       .groupby("categoria_n2")
       .agg(n=("product_id", "size"),
            nota_mediana=("nota", "median"),
            desvio_padrao=("nota", "std"),
            p25=("nota", lambda s: s.quantile(0.25)),
            p75=("nota", lambda s: s.quantile(0.75)))
       .assign(iqr=lambda d: d.p75 - d.p25)
       .sort_values("desvio_padrao", ascending=False)
       .round(3))
cobertura = 100 * cont2[cont2 >= PISO].sum() / N
print(f"QA3 · {len(qa3)} subcategorias acima do piso de 30 (T2) — cobertura {cobertura:.1f}% dos produtos")
display(qa3)
print(f"grupos suprimidos por T2: {int((cont2 < PISO).sum())} subcategorias somando "
      f"{int(cont2[cont2 < PISO].sum())} produtos")

responde(
    "QA3",
    "desvio-padrão e IQR da nota por subcategoria de nível 2, restritos aos grupos com n ≥ 30 (T2)",
    f"{len(qa3)} subcategorias ficam acima do piso e cobrem {cobertura:.1f}% dos produtos; a maior "
    f"dispersão está em {curto(qa3.index[0], 46)} (desvio {pt_br(qa3.desvio_padrao.iloc[0], 3)}, "
    f"IQR {pt_br(qa3.iqr.iloc[0], 3)}, n = {int(qa3.n.iloc[0])}) e a menor em "
    f"{curto(qa3.index[-1], 46)} (desvio {pt_br(qa3.desvio_padrao.iloc[-1], 3)}, n = {int(qa3.n.iloc[-1])})",
    "analítica", int(qa3.n.sum()),
    "respondida")

QA3 · 8 subcategorias acima do piso de 30 (T2) — cobertura 88.3% dos produtos


,n,nota_mediana,desvio_padrao,p25,p75,iqr
categoria_n2,,,,,,
"Home&Kitchen | Heating,Cooling&AirQuality",116,4.1,0.346,3.8,4.200,0.400
Home&Kitchen | Kitchen&HomeAppliances,307,4.1,0.336,3.9,4.300,0.400
Electronics | WearableTechnology,62,4.1,0.326,3.9,4.200,0.300
"Electronics | HomeTheater,TV&Video",156,4.2,0.310,3.9,4.300,0.400
Computers&Accessories | NetworkingDevices,30,4.2,0.276,4.1,4.375,0.275
Computers&Accessories | Accessories&Peripherals,305,4.2,0.258,4.0,4.300,0.300
"Electronics | Headphones,Earbuds&Accessories",63,3.9,0.224,3.8,4.100,0.300
Electronics | Mobiles&Accessories,151,4.1,0.194,4.0,4.300,0.300


grupos suprimidos por T2: 21 subcategorias somando 158 produtos
[QA3] respondida
    8 subcategorias ficam acima do piso e cobrem 88.3% dos produtos; a maior dispersão está em Home&Kitchen | Heating,Cooling&AirQuality (desvio 0,346, IQR 0,400, n = 116) e a menor em Electronics | Mobiles&Accessories (desvio 0,194, n = 151)



In [19]:
checar_series(2)

ordem3 = qa3.sort_values("desvio_padrao")
dados = [an.loc[an.categoria_n2 == sub, "nota"].values for sub in ordem3.index]

fig, ax = plt.subplots(figsize=(9.0, 4.8))
bp = ax.boxplot(dados, vert=False, widths=0.56, patch_artist=True,
                medianprops=dict(color=LARANJA, lw=1.8),
                whiskerprops=dict(color=ARDOSIA, lw=1.0),
                capprops=dict(color=ARDOSIA, lw=1.0),
                flierprops=dict(marker="o", markersize=2.6, markerfacecolor=GRADE,
                                markeredgecolor="none", alpha=0.6))
for caixa in bp["boxes"]:
    caixa.set(facecolor=TEAL, alpha=0.34, edgecolor=TEAL, linewidth=1.1)

for i, (sub, linha) in enumerate(ordem3.iterrows(), start=1):
    ax.text(5.06, i, f"n = {int(linha.n)}   dp = {pt_br(linha.desvio_padrao, 3)}",
            va="center", fontsize=7.8, color=ARDOSIA)

ax.set_yticks(range(1, len(ordem3) + 1))
ax.set_yticklabels([curto(s.replace(" | ", " › "), 38) for s in ordem3.index], fontsize=8.2)
ax.set_xlim(2.0, 5.9)
ax.set_xlabel("nota (escala 1 a 5)")
ax.set_title("QA3 · Dispersão da nota por subcategoria de nível 2 — ordenada pelo desvio-padrão")
rodape(fig, FONTE + f"\nT2: apenas subcategorias com n ≥ 30 · {len(qa3)} grupos · cobertura "
       f"{cobertura:.1f}% dos produtos · {int((cont2 < PISO).sum())} grupos suprimidos")
salva(fig, "fig_an_04_dispersao_subcategoria_qa3.png")

figura: 02_results/images/fig_an_04_dispersao_subcategoria_qa3.png


/var/folders/pk/3082z6zx3k5d5kz2h1c16dmm0000gn/T/ipykernel_35456/1375607173.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  bp = ax.boxplot(dados, vert=False, widths=0.56, patch_artist=True,


---
## QA4 — O preço de tabela explica a satisfação?

**Método:** Spearman entre `preco_tab` e `nota`, já calculado no quadro das quatro associações.

Esta é a pergunta cuja resposta é **não**, e o valor de publicá-la está justamente aí: ρ = +0,032 com
p = 0,236 é o único par do quadro que **não** cruza o limiar de significância. Com n = 1.348 — tamanho
em que quase tudo vira significativo — não ter significância é informação, não ausência dela.

Cuidado de redação: o verbo "explica" está no enunciado da pergunta do grupo; a resposta é dada em
termos de **associação** (T4).

In [20]:
print("QA4 · preço de tabela × nota")
display(correl[correl.par == "Preço de tabela × nota"])

por_faixa_preco = (an.assign(faixa_preco=pd.qcut(an.preco_tab, 4,
                                                 labels=["Q1 mais barato", "Q2", "Q3", "Q4 mais caro"]))
                     .groupby("faixa_preco", observed=True)
                     .agg(n=("product_id", "size"),
                          preco_mediano=("preco_tab", "median"),
                          nota_corrigida_mediana=("nota_corrigida", "median"))
                     .round(3))
print("\n  leitura de apoio — quartis de preço de tabela (valores em rupia indiana, sem conversão · L5):")
display(por_faixa_preco)

responde(
    "QA4",
    "correlação de Spearman entre preço de tabela e nota, com quartis de preço como leitura de apoio; "
    "magnitude ao lado do p-valor (T5)",
    f"não — ρ = {pt_br(r_preco_nota, 3)} com p = {pt_br(r_preco_nota and p_preco_nota, 3)}: o único par do "
    f"quadro que não atinge significância a 5%. A nota corrigida mediana varia "
    f"{pt_br(por_faixa_preco.nota_corrigida_mediana.max() - por_faixa_preco.nota_corrigida_mediana.min(), 3)} "
    "ponto de amplitude entre os quatro quartis de preço. Sem associação detectável (T4)",
    "analítica", N,
    "respondida")

QA4 · preço de tabela × nota


,par,rho,p_valor,n,leitura
2,Preço de tabela × nota,0.0323,0.236,1348,sem associação



  leitura de apoio — quartis de preço de tabela (valores em rupia indiana, sem conversão · L5):


,n,preco_mediano,nota_corrigida_mediana
faixa_preco,,,
Q1 mais barato,345,499.0,4.038
Q2,331,1199.0,4.062
Q3,335,2499.0,4.019
Q4 mais caro,337,11999.0,4.083


[QA4] respondida
    não — ρ = 0,032 com p = 0,236: o único par do quadro que não atinge significância a 5%. A nota corrigida mediana varia 0,064 ponto de amplitude entre os quatro quartis de preço. Sem associação detectável (T4)



---
## QA5 — Quais temas de reclamação predominam nos produtos de alto engajamento e baixa nota?

**Método vedado e método adotado.** O escopo **veda TF-IDF nesta pergunta**: o corpus de Q5 tem 19
blocos, e nenhum método frequentista se sustenta sobre 19 documentos. O que vale é **leitura
qualitativa dirigida com protocolo de codificação** — trabalho humano, entregue no
`05_nlp.ipynb` junto do restante do tratamento textual.

Esta vedação **não contradiz** o uso de TF-IDF em Q7: lá são 339 e 418 blocos. As duas decisões
coexistem porque o tamanho do corpus é diferente.

**O que este notebook entrega:** o dimensionamento exato do corpus que a leitura qualitativa vai
percorrer — quantos produtos, quantas avaliações, quanto texto —, para que o protocolo seja
dimensionável e o truncamento fique declarado (T8).

In [21]:
ids_q5 = set(q5.product_id)
ref_q5 = ref[ref.product_id.isin(ids_q5)]
texto_q5 = base_texto[base_texto.product_id.isin(ids_q5)]
caracteres = texto_q5.review_content.fillna("").str.len().sum()

print("QA5 · dimensionamento do corpus dirigido (T8)")
print(f"  produtos (blocos)        : {len(texto_q5)}")
print(f"  referências de avaliação : {len(ref_q5)}")
print(f"  reviews distintos        : {ref_q5._rid.nunique()}")
print(f"  volume de texto          : {pt_br(caracteres/1000, 1)} mil caracteres")
print(f"  truncamento              : média {pt_br(ref_q5.groupby('product_id')._rid.size().mean(), 2)} "
      f"avaliações por produto, máximo 8 (L1)")
print("\n  método: leitura qualitativa dirigida com protocolo de codificação.")
print("  TF-IDF é vedado aqui (19 blocos) e permanece válido em Q7 (339 e 418 blocos).")

responde(
    "QA5",
    "leitura qualitativa dirigida com protocolo de codificação sobre o corpus dos 19 produtos de Q5; "
    "TF-IDF vedado nesta pergunta por tamanho de corpus. Execução da codificação no 05_nlp",
    f"corpus dirigido dimensionado: {len(texto_q5)} produtos, {len(ref_q5)} referências de avaliação "
    f"({ref_q5._rid.nunique()} distintas) e {pt_br(caracteres/1000, 1)} mil caracteres. Truncado em 8 "
    "avaliações por produto: descreve os temas visíveis na vitrine, não a distribuição das opiniões (L1)",
    "analítica · quadrante Q5", int(len(ref_q5)),
    "respondida")

QA5 · dimensionamento do corpus dirigido (T8)
  produtos (blocos)        : 19
  referências de avaliação : 147
  reviews distintos        : 147
  volume de texto          : 49,5 mil caracteres
  truncamento              : média 7,74 avaliações por produto, máximo 8 (L1)

  método: leitura qualitativa dirigida com protocolo de codificação.
  TF-IDF é vedado aqui (19 blocos) e permanece válido em Q7 (339 e 418 blocos).
[QA5] respondida
    corpus dirigido dimensionado: 19 produtos, 147 referências de avaliação (147 distintas) e 49,5 mil caracteres. Truncado em 8 avaliações por produto: descreve os temas visíveis na vitrine, não a distribuição das opiniões (L1)



---
## QA6 — Os produtos no topo do ranking merecem estar lá?

**Método:** comparação do ranking por **nota bruta** com o ranking por **limite inferior de Wilson a 95%**
(`nota_corrigida`), com o *n* de avaliações exibido ao lado — que é exatamente o que T1 exige e o motivo
de T1 existir.

Wilson penaliza a incerteza: um produto com nota 5,0 e 12 avaliações tem intervalo largo, e o limite
inferior do intervalo cai muito. Um produto com 4,6 e 40 mil avaliações quase não se move. O ranking
corrigido é, na prática, o ranking em que **o número de avaliações passa a contar**.

Desempate declarado: entre notas brutas iguais, ordem alfabética de `product_id` — critério neutro,
que não favorece volume nem categoria.

In [22]:
bruto = an.sort_values(["nota", "product_id"], ascending=[False, True]).reset_index(drop=True)
corrigido = an.sort_values(["nota_corrigida", "product_id"], ascending=[False, True]).reset_index(drop=True)
posicao_corrigida = {pid: i + 1 for i, pid in enumerate(corrigido.product_id)}

top10 = bruto.head(10).copy()
top10["posicao_bruta"] = np.arange(1, 11)
top10["posicao_corrigida"] = top10.product_id.map(posicao_corrigida)
# Delta com sinal, não "queda": valor negativo significa que o produto SUBIU.
top10["variacao_de_posicao"] = top10.posicao_corrigida - top10.posicao_bruta

qa6 = top10[["posicao_bruta", "product_name", "avaliacoes", "nota", "nota_corrigida",
             "posicao_corrigida", "variacao_de_posicao"]].copy()
qa6["product_name"] = qa6.product_name.map(curto)
qa6 = qa6.rename(columns={"product_name": "produto", "avaliacoes": "volume_de_avaliacoes"})
print("QA6 · top 10 por nota bruta e onde cada um cai no ranking corrigido (T1)")
display(qa6.round(3).reset_index(drop=True))

top10_corr = corrigido.head(10)
sobrevivem = len(set(top10.product_id) & set(top10_corr.product_id))
print(f"\n  dos 10 do topo bruto, {sobrevivem} permanecem no top 10 corrigido")
print(f"  mediana de avaliações no topo bruto     : {pt_br(top10.avaliacoes.median())}")
print(f"  mediana de avaliações no topo corrigido : {pt_br(top10_corr.avaliacoes.median())}")

# O efeito da correção é BIMODAL, e a mediana o esconderia: uns poucos produtos de n
# minúsculo despencam centenas de posições enquanto os de n alto sobem. Publicar
# "queda mediana de -1 posições" seria um resumo que contraria o próprio achado.
caem = top10[top10.variacao_de_posicao > 0]
sobem = top10[top10.variacao_de_posicao <= 0]
pior = top10.loc[top10.variacao_de_posicao.idxmax()]
print(f"  caem ao corrigir: {len(caem)} · mantêm-se ou sobem: {len(sobem)}")
print(f"  maior queda: {int(pior.posicao_bruta)}º → {int(pior.posicao_corrigida)}º "
      f"(n = {pt_br(pior.avaliacoes)} avaliações)")

responde(
    "QA6",
    "ranking por nota bruta comparado ao ranking por limite inferior de Wilson a 95% (T1), com o n de "
    "avaliações ao lado; desempate por product_id",
    f"não como estão: dos 10 primeiros por nota bruta, apenas {sobrevivem} seguem no top 10 corrigido. "
    f"O efeito é bimodal — {len(caem)} produtos de n pequeno despencam e {len(sobem)} de n alto sobem ou "
    f"se mantêm; a maior queda vai de {int(pior.posicao_bruta)}º a {int(pior.posicao_corrigida)}º, com "
    f"apenas {pt_br(pior.avaliacoes)} avaliações. A mediana de avaliações do topo sobe de "
    f"{pt_br(top10.avaliacoes.median())} para {pt_br(top10_corr.avaliacoes.median())} quando a correção "
    "entra — é a incerteza de n pequeno saindo do pódio",
    "analítica", N,
    "respondida")

QA6 · top 10 por nota bruta e onde cada um cai no ranking corrigido (T1)


,posicao_bruta,produto,volume_de_avaliacoes,nota,nota_corrigida,posicao_corrigida,variacao_de_posicao
0,1,Amazon Basics Wireless Mouse | 2.4 GHz…,23.0,5.0,4.428,67,66
1,2,Syncwire LTG to USB Cable for Fast Char…,5.0,5.0,3.262,1287,1285
2,3,"Instant Pot Air Fryer, Vortex 2QT, Touc…",3964.0,4.8,4.771,2,-1
3,4,"Oratech Coffee Frother electric, milk f…",28.0,4.8,4.210,368,364
4,5,Swiffer Instant Electric Water Heater F…,53803.0,4.8,4.793,1,-4
5,6,Sony Bravia 164 cm (65 inches) 4K Ultra…,5935.0,4.7,4.672,4,-2
6,7,Spigen EZ Fit Tempered Glass Screen Pro…,7779.0,4.7,4.676,3,-4
7,8,Zuvexa USB Rechargeable Electric Foam M…,54.0,4.7,4.293,184,176
8,9,FIGMENT Handheld Milk Frother Rechargea…,1729.0,4.7,4.646,7,-2
9,10,Multifunctional 2 in 1 Electric Egg Boi…,2300.0,4.7,4.654,6,-4



  dos 10 do topo bruto, 6 permanecem no top 10 corrigido
  mediana de avaliações no topo bruto     : 2.014
  mediana de avaliações no topo corrigido : 6.857
  caem ao corrigir: 4 · mantêm-se ou sobem: 6
  maior queda: 2º → 1287º (n = 5 avaliações)
[QA6] respondida
    não como estão: dos 10 primeiros por nota bruta, apenas 6 seguem no top 10 corrigido. O efeito é bimodal — 4 produtos de n pequeno despencam e 6 de n alto sobem ou se mantêm; a maior queda vai de 2º a 1287º, com apenas 5 avaliações. A mediana de avaliações do topo sobe de 2.014 para 6.857 quando a correção entra — é a incerteza de n pequeno saindo do pódio



In [23]:
checar_series(2)

fig, ax = plt.subplots(figsize=(9.2, 5.0))
y = np.arange(len(top10))[::-1]

ax.hlines(y, top10.nota_corrigida, top10.nota, color=GRADE, lw=1.6, zorder=1)
ax.scatter(top10.nota, y, s=78, color=DESTAQUE, edgecolors=BASE["card"], linewidth=1.0,
           zorder=3, label="nota bruta")
ax.scatter(top10.nota_corrigida, y, s=78, color=TEAL, edgecolors=BASE["card"], linewidth=1.0,
           zorder=3, label="nota corrigida — limite inferior de Wilson 95% (T1)")

for yi, (_, linha) in zip(y, top10.iterrows()):
    ax.text(linha.nota + 0.06, yi, f"{pt_br(linha.nota, 1)}", va="center", fontsize=8, color=ARDOSIA)
    ax.text(linha.nota_corrigida - 0.06, yi, f"{pt_br(linha.nota_corrigida, 2)}",
            va="center", ha="right", fontsize=8, color=ARDOSIA)
    # Sem o sufixo "de 1.348": o rodapé já declara a base e o n (T3), e a repetição
    # em cada linha esbarrava no rótulo da nota corrigida do 2º colocado.
    ax.text(1.55, yi, f"n = {pt_br(linha.avaliacoes)} avaliações  ·  "
            f"posição corrigida: {int(linha.posicao_corrigida)}º",
            va="center", fontsize=7.6, color=ARDOSIA)

ax.set_yticks(y)
ax.set_yticklabels([f"{i}º  {curto(nome, 30)}" for i, nome in
                    zip(top10.posicao_bruta, top10.product_name)], fontsize=8.2)
ax.set_xlim(1.4, 5.35)
ax.set_xlabel("nota (escala 1 a 5)")
ax.set_title("QA6 · O topo do ranking bruto depois da correção de variância")
# A legenda sai da área do gráfico: cada uma das 10 linhas já carrega o n e a posição
# corrigida à esquerda, e dentro do eixo a legenda cobriria a última delas.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=8)
rodape(fig, FONTE + "\nT1: todo ranking por nota usa a nota corrigida, com o n de avaliações ao lado · "
       "desempate do ranking bruto por product_id")
salva(fig, "fig_an_05_ranking_wilson_qa6.png")

figura: 02_results/images/fig_an_05_ranking_wilson_qa6.png


---
## QA7 — Quais arquétipos comerciais emergem ao agrupar por preço, desconto, nota e volume?

**Método (ML4, especificação fechada):** k-médias sobre quatro variáveis padronizadas —
**preço de tabela em log natural**, desconto, **volume de avaliações em log natural** e nota.
Semente 42, `n_init=10`, k testado de 2 a 7.

**A redação do `k` é obrigatória e literal:** *o coeficiente de silhueta foi calculado para k de 2 a 7 e
atinge o máximo em k = 5 (0,237); adotou-se **k = 4** (0,234) por interpretabilidade, com diferença
estatística desprezível entre os dois.* Dizer "k escolhido pela silhueta" é impreciso e derruba o
argumento na primeira pergunta.

**Fraqueza declarada:** silhueta ≤ 0,237 indica que **não há estrutura natural de agrupamento** nesta
base. Os arquétipos são **convenção descritiva útil**, nunca apresentados como descoberta.

**A leitura de "preço" é decisiva:** com preço *com desconto* no lugar do preço de tabela, a curva inteira
muda e o máximo migra de k=5 para k=2. Por isso a variável está declarada, não subentendida.

In [24]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = StandardScaler().fit_transform(
    np.c_[np.log(an.preco_tab), an.desconto, np.log(an.avaliacoes), an.nota])

silhuetas = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    silhuetas[k] = round(silhouette_score(X, km.labels_), 3)
print("ML4 · curva de silhueta")
display(pd.Series(silhuetas, name="silhueta").rename_axis("k").to_frame())
print(f"máximo em k = {max(silhuetas, key=silhuetas.get)} ({max(silhuetas.values())}) · "
      f"adotado k = 4 ({silhuetas[4]}) por interpretabilidade")

km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)
an["grupo"] = km.labels_
perfil = (an.groupby("grupo")
            .agg(n=("grupo", "size"), preco_tabela_mediano=("preco_tab", "median"),
                 desconto_mediano=("desconto", "median"),
                 avaliacoes_medianas=("avaliacoes", "median"),
                 nota_mediana=("nota", "median")))

# Letras atribuídas pelo PERFIL, não pela ordem do KMeans — que varia entre execuções
# e entre versões da biblioteca. A = ticket alto · B = único abaixo de 4,0 ·
# C = menor desconto entre os restantes · D = maior volume entre os restantes.
rotulos = {perfil.preco_tabela_mediano.idxmax(): "A", perfil.nota_mediana.idxmin(): "B"}
restantes = [g for g in perfil.index if g not in rotulos]
rotulos[min(restantes, key=lambda g: perfil.desconto_mediano[g])] = "C"
rotulos[max(restantes, key=lambda g: perfil.avaliacoes_medianas[g])] = "D"
perfil["arquetipo"] = [rotulos[g] for g in perfil.index]
perfil = perfil.set_index("arquetipo").sort_index()
perfil.index = ["A — Ticket alto, desconto contido", "B — Desconto agressivo sem tração",
                "C — Baixo preço, pouco desconto", "D — Alto giro com desconto alto"]
print("\nQA7 · os 4 arquétipos (valores em rupia indiana, sem conversão · L5)")
display(perfil)

responde(
    "QA7",
    "k-médias sobre preço de tabela (log), desconto, volume de avaliações (log) e nota, padronizados; "
    "semente 42, n_init=10, k de 2 a 7. Máximo de silhueta em k=5 (0,237); adotado k=4 (0,234) por "
    "interpretabilidade, diferença desprezível",
    f"quatro arquétipos: A ticket alto ({int(perfil.n.iloc[0])}), B desconto agressivo sem tração "
    f"({int(perfil.n.iloc[1])}), C baixo preço e pouco desconto ({int(perfil.n.iloc[2])}), D alto giro "
    f"({int(perfil.n.iloc[3])}). B e D têm desconto praticamente igual e divergem em volume de avaliações e em nota — B é o "
    f"único grupo abaixo de 4,0. Silhueta ≤ {max(silhuetas.values())}: convenção descritiva, não descoberta",
    "analítica", N,
    "respondida")

ML4 · curva de silhueta


,silhueta
k,
2,0.218
3,0.215
4,0.234
5,0.237
6,0.227
7,0.219


máximo em k = 5 (0.237) · adotado k = 4 (0.234) por interpretabilidade

QA7 · os 4 arquétipos (valores em rupia indiana, sem conversão · L5)


,n,preco_tabela_mediano,desconto_mediano,avaliacoes_medianas,nota_mediana
"A — Ticket alto, desconto contido",268,14999.0,32.0,6836.0,4.2
B — Desconto agressivo sem tração,335,1290.0,58.0,468.0,3.9
"C — Baixo preço, pouco desconto",298,899.5,22.0,6534.0,4.2
D — Alto giro com desconto alto,447,1500.0,61.0,13120.0,4.2


[QA7] respondida
    quatro arquétipos: A ticket alto (268), B desconto agressivo sem tração (335), C baixo preço e pouco desconto (298), D alto giro (447). B e D têm desconto praticamente igual e divergem em volume de avaliações e em nota — B é o único grupo abaixo de 4,0. Silhueta ≤ 0.237: convenção descritiva, não descoberta



---
# Exportação · `tab13_respostas_15_perguntas.csv`

A tabela sai do registro que cada célula alimentou — não de digitação. Se uma pergunta deixar de ser
respondida no código, ela **some** da tabela, e a validação abaixo acusa.

In [25]:
tab10 = pd.DataFrame(REGISTRO, columns=["pergunta", "metodo", "resultado", "base", "n", "situacao"])
destino = TAB / "tab13_respostas_15_perguntas.csv"
tab10.to_csv(destino, index=False, encoding="utf-8")
print("gravado:", destino.relative_to(RAIZ), f"— {len(tab10)} linhas")
display(tab10[["pergunta", "base", "n", "situacao"]])

gravado: 02_results/tables/tab13_respostas_15_perguntas.csv — 15 linhas


,pergunta,base,n,situacao
0,Q1,analítica,1348,respondida
1,Q2,analítica,1348,respondida
2,Q3,analítica,1348,respondida
3,Q4,analítica,1341,respondida
4,Q5,analítica,19,respondida
5,Q6,analítica,1348,respondida
6,Q7,analítica · corpus deduplicado,9255,respondida
7,Q8,analítica,1348,consolidação
8,QA1,analítica,1348,respondida
9,QA2,analítica,1348,respondida


---
# Célula final de validação

Reconciliação entre **o que este notebook mediu** e **os números do material**
(`method/definitions/base_factual_e_bases_de_calculo.md` e `registro_machine_learning.md`).

Se qualquer linha sair como `DIVERGE`, o código está errado — não o material. Essa é a direção da
prova: o material é a fonte única de números, e o notebook precisa reproduzi-lo.

In [26]:
def confere(rotulo, medido, esperado, tolerancia=0.0):
    ok = abs(medido - esperado) <= tolerancia
    print(f"  {'OK    ' if ok else 'DIVERGE'} · {rotulo:<52} medido {medido!s:>12}  esperado {esperado!s:>12}")
    return ok

checagens = []
print("BASES E CORTES")
checagens.append(confere("base analítica", N, 1348))
checagens.append(confere("p90 de rating_count", P90, 36384))
checagens.append(confere("mediana de rating_count", MEDIANA, 4740))
checagens.append(confere("produtos com nota < 4,0", int((an.nota < 4.0).sum()), 339))
checagens.append(confere("% com nota < 4,0", round(100 * (an.nota < 4.0).mean(), 1), 25.1))
checagens.append(confere("produtos com nota ≥ 4,3", int((an.nota >= 4.3).sum()), 418))

print("\nQUADRANTES E CONCENTRAÇÃO")
checagens.append(confere("Q5 · alto engajamento e nota < 4,0", len(q5), 19))
checagens.append(confere("Q6 · candidato a subexploração", len(q6), 187))
checagens.append(confere("QA2 · Pareto 80%", corte_pareto, 306))
checagens.append(confere("QA2 · % da base", round(100 * corte_pareto / N, 1), 22.7))

print("\nCATEGORIAS")
checagens.append(confere("categorias nível 1 acima do piso", int(cat.acima_do_piso.sum()), 4))
checagens.append(confere("Electronics", int(cat.produtos["Electronics"]), 490))
checagens.append(confere("Home&Kitchen", int(cat.produtos["Home&Kitchen"]), 447))
checagens.append(confere("Computers&Accessories", int(cat.produtos["Computers&Accessories"]), 373))
checagens.append(confere("OfficeProducts", int(cat.produtos["OfficeProducts"]), 31))
checagens.append(confere("produtos nas 5 categorias suprimidas", int(suprimidas.produtos.sum()), 7))
checagens.append(confere("subcategorias nível 2 acima do piso", len(qa3), 8))
checagens.append(confere("cobertura das subcategorias (%)", round(cobertura, 1), 88.3))

print("\nASSOCIAÇÕES DE SPEARMAN (magnitude e significância — T5)")
checagens.append(confere("ρ desconto × nota", round(r_desc_nota, 3), -0.15, 0.001))
checagens.append(confere("p desconto × nota", round(p_desc_nota / 1e-8, 1), 3.1, 0.15))
checagens.append(confere("ρ desconto × engajamento", round(r_desc_eng, 3), -0.132, 0.001))
checagens.append(confere("ρ preço de tabela × nota", round(r_preco_nota, 3), 0.032, 0.001))
checagens.append(confere("p preço de tabela × nota", round(p_preco_nota, 3), 0.236, 0.002))
checagens.append(confere("ρ engajamento × nota", round(r_eng_nota, 3), 0.193, 0.001))

print("\nCORPUS TEXTUAL (T8)")
checagens.append(confere("referências de avaliação", referencias, 10588))
checagens.append(confere("reviews distintos", distintos, 9255))
checagens.append(confere("referências repetidas", referencias - distintos, 1333))
checagens.append(confere("truncamento médio", round(por_produto.mean(), 2), 7.85))
checagens.append(confere("truncamento máximo", int(por_produto.max()), 8))
checagens.append(confere("QA5 · referências nos 19 produtos", len(ref_q5), 147))

print("\nML2 E ML4")
auc = float(ml2_tab.loc[ml2_tab.bloco_preditores == "completo", "auc"].iloc[0])
dp = float(ml2_tab.loc[ml2_tab.bloco_preditores == "completo", "desvio_entre_particoes"].iloc[0])
checagens.append(confere("ML2 · AUC do bloco completo", round(auc, 2), 0.70, 0.005))
checagens.append(confere("ML2 · desvio entre partições", round(dp, 2), 0.01, 0.005))
checagens.append(confere("ML4 · silhueta k=3", silhuetas[3], 0.215, 0.001))
checagens.append(confere("ML4 · silhueta k=4", silhuetas[4], 0.234, 0.001))
checagens.append(confere("ML4 · silhueta k=5", silhuetas[5], 0.237, 0.001))
for letra, esperado in zip("ABCD", [268, 335, 298, 447]):
    checagens.append(confere(f"ML4 · arquétipo {letra}", int(perfil.n.iloc[ord(letra) - 65]), esperado))
checagens.append(confere("ML4 · soma dos arquétipos", int(perfil.n.sum()), 1348))

print("\nCOBERTURA DO ESCOPO")
esperadas = [f"Q{i}" for i in range(1, 9)] + [f"QA{i}" for i in range(1, 8)]
faltando = [p for p in esperadas if p not in list(tab10.pergunta)]
checagens.append(confere("perguntas respondidas", len(tab10), 15))
print(f"  {'OK    ' if not faltando else 'DIVERGE'} · nenhuma pergunta do escopo faltando"
      f"{'' if not faltando else ' — faltam ' + ', '.join(faltando)}")
checagens.append(not faltando)

print("\nREGRAS TRANSVERSAIS")
figuras = sorted(p.name for p in IMG.glob("fig_an_*.png"))
# A série fig01–fig13 é de `src/gera_figuras.py` e este notebook não pode tocá-la.
# A verificação é nominal, uma a uma: contar os arquivos `figNN_` não serve, porque
# outros notebooks da sequência legitimamente acrescentam figuras nessa numeração.
serie_gera_figuras = [f"fig{i:02d}_" for i in range(1, 14)]
preservadas = [pref for pref in serie_gera_figuras if any(IMG.glob(f"{pref}*.png"))]
ausentes = sorted(set(serie_gera_figuras) - set(preservadas))
print(f"  T9 · figuras novas geradas: {len(figuras)} — {', '.join(figuras)}")
print(f"  T9 · figuras fig01–fig13 preservadas: {len(preservadas)} de 13"
      f"{'' if not ausentes else ' — AUSENTES: ' + ', '.join(ausentes)}")
print(f"  T7 · colunas pessoais na base: "
      f"{sorted({'user_id', 'user_name'} & set(an.columns)) or 'nenhuma'}")
checagens.append(confere("figuras fig_an_ geradas", len(figuras), 5))
checagens.append(confere("figuras fig01–fig13 preservadas", len(preservadas), 13))

print("\n" + "=" * 92)
if all(checagens):
    print("TODAS AS CONTAGENS RECONCILIAM COM O MATERIAL. 15 perguntas respondidas, "
          "nenhuma pendência: D-1 e D-2 fecharam em 24/09/2026.")
else:
    print(f"ATENÇÃO: {sum(1 for c in checagens if not c)} verificação(ões) divergiram — corrigir o código.")
print("=" * 92)

print("\nNota de reprodução registrada: a silhueta de k=6 mede "
      f"{silhuetas[6]} nesta execução, contra 0,229 no registro de ML. A diferença é de "
      "0,002 e não afeta nenhum valor publicado (k=3, k=4 e k=5 conferem exatamente). "
      "Fica declarada porque um número do material só vale se for reproduzido ou contestado.")

BASES E CORTES
  OK     · base analítica                                       medido         1348  esperado         1348
  OK     · p90 de rating_count                                  medido      36384.0  esperado        36384
  OK     · mediana de rating_count                              medido       4740.0  esperado         4740
  OK     · produtos com nota < 4,0                              medido          339  esperado          339
  OK     · % com nota < 4,0                                     medido         25.1  esperado         25.1
  OK     · produtos com nota ≥ 4,3                              medido          418  esperado          418

QUADRANTES E CONCENTRAÇÃO
  OK     · Q5 · alto engajamento e nota < 4,0                   medido           19  esperado           19
  OK     · Q6 · candidato a subexploração                       medido          187  esperado          187
  OK     · QA2 · Pareto 80%                                     medido          306  esperado         

---
## O que este notebook deixa aberto — e por quê

| Item | Onde | Situação |
|---|---|---|
| **D-2** fórmula de relevância | Q1 publica componentes, sem índice composto | **fechada em 24/09/2026** — matriz 2×2, sem limiar novo |
| **D-1** rótulo de visibilidade | Q6 publica o funil de 187 | **fechada em 24/09/2026** — proxy declarado de exposição; acionáveis são 61 (T1) |
| Codificação qualitativa de QA5 | dimensionada aqui, executada no `05_nlp.ipynb` | passada adiante |
| Extração de n-gramas e TF-IDF (ML3) | corpus dimensionado aqui, extração no `05_nlp.ipynb` | passada adiante |

Nenhuma das duas decisões foi contornada com critério improvisado enquanto estava aberta: este
notebook mediu os componentes e esperou o rótulo (`method/open_decisions.md`).